# 🔥 Activity 08: Linear Regression with PyTorch

**Track:** Linear Regression  
**Level:** Advanced

---

## 📖 What You Will Learn
- PyTorch fundamentals: tensors, autograd, device management
- `nn.Module` — build custom models
- Custom training loop: forward → loss → backward → optimizer.step()
- Data loading with `TensorDataset` and `DataLoader`
- Loss tracking and validation
- Compare with sklearn and TensorFlow

---

## 🧠 Concept: PyTorch vs TensorFlow

| | PyTorch | TensorFlow/Keras |
|---|---|---|
| **Style** | Explicit custom loop | High-level `model.fit()` |
| **Control** | Full control | Less boilerplate |
| **Research** | Dominant in academia | Dominant in production |
| **Debugging** | Easier (Pythonic) | Harder (graph mode) |

> **The PyTorch training loop:** Forward pass → Compute loss → Backprop → Update parameters

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

import warnings; warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')

## 📊 Data Preparation

In [ ]:
housing = fetch_california_housing(as_frame=True)
X = housing.frame.drop(columns=['MedHouseVal'])
y = housing.frame['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler  = StandardScaler()
X_tr_np = scaler.fit_transform(X_train).astype(np.float32)
X_te_np = scaler.transform(X_test).astype(np.float32)
y_tr_np = y_train.values.astype(np.float32).reshape(-1, 1)
y_te_np = y_test.values.astype(np.float32).reshape(-1, 1)

# Convert to PyTorch tensors
X_tr_t = torch.FloatTensor(X_tr_np).to(DEVICE)
X_te_t = torch.FloatTensor(X_te_np).to(DEVICE)
y_tr_t = torch.FloatTensor(y_tr_np).to(DEVICE)
y_te_t = torch.FloatTensor(y_te_np).to(DEVICE)

# DataLoader for mini-batch training
train_dataset = TensorDataset(X_tr_t, y_tr_t)
train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f'Train batches: {len(train_loader)}  Batch size: 64')

## 🏗️ Model 1 — Linear Regression (`nn.Linear`)

In [ ]:
class LinearRegressionPyTorch(nn.Module):
    """
    Single linear layer = Linear Regression.
    WHY nn.Module? It tracks parameters automatically for backprop.
    """
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)  # y = Wx + b

    def forward(self, x):
        return self.linear(x)


linear_model = LinearRegressionPyTorch(input_dim=X_tr_np.shape[1]).to(DEVICE)
print(linear_model)
print(f'\nParameters: {sum(p.numel() for p in linear_model.parameters())}')

## 🔄 Custom Training Loop

### 🧠 Concept: The PyTorch Loop

```python
for epoch in range(epochs):
    for X_batch, y_batch in dataloader:
        optimizer.zero_grad()    # Clear gradients from previous step
        y_pred = model(X_batch)  # Forward pass
        loss   = criterion(y_pred, y_batch)  # Compute loss
        loss.backward()          # Backpropagation (compute gradients)
        optimizer.step()         # Update parameters
```

> **⚠️ COMMON ERROR:** Forgetting `optimizer.zero_grad()` causes gradients to **accumulate** across batches → wrong updates → divergence.

In [ ]:
def train_model(model, train_loader, X_val, y_val,
                epochs=200, lr=0.001, patience=20):
    """
    Generic PyTorch training loop with:
    - Mini-batch training
    - Validation tracking
    - Early stopping
    - Best model checkpoint
    """
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, verbose=False
    )

    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_state    = None
    no_improve    = 0

    for epoch in range(epochs):
        # ── Training phase ────────────────────────────────────────────────────
        model.train()
        batch_losses = []
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()              # CRITICAL: reset gradients
            y_pred = model(X_batch)
            loss   = criterion(y_pred, y_batch)
            loss.backward()                    # compute gradients
            optimizer.step()                   # update weights
            batch_losses.append(loss.item())

        train_loss = np.mean(batch_losses)
        train_losses.append(train_loss)

        # ── Validation phase ──────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():                  # no gradient computation needed
            val_pred = model(X_val)
            val_loss = criterion(val_pred, y_val).item()
        val_losses.append(val_loss)
        scheduler.step(val_loss)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)   # restore best weights
    return train_losses, val_losses


# Use a portion of training set as validation
n_train = len(X_tr_t)
n_val   = int(n_train * 0.2)
X_val_t, y_val_t = X_tr_t[-n_val:], y_tr_t[-n_val:]

val_dataset    = TensorDataset(X_tr_t[:-n_val], y_tr_t[:-n_val])
train_loader_v = DataLoader(val_dataset, batch_size=64, shuffle=True)

train_losses, val_losses = train_model(
    linear_model, train_loader_v, X_val_t, y_val_t,
    epochs=300, lr=0.001, patience=20
)

In [ ]:
# ── Loss curves ───────────────────────────────────────────────────────────────
plt.figure(figsize=(9, 4))
plt.plot(train_losses, label='Train MSE')
plt.plot(val_losses,   label='Val MSE')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('PyTorch Linear Model — Loss Curves')
plt.legend()
plt.yscale('log')
plt.tight_layout()
plt.show()

# Evaluate
linear_model.eval()
with torch.no_grad():
    y_pred_t = linear_model(X_te_t).cpu().numpy().flatten()

r2_pt_lin = r2_score(y_te_np.flatten(), y_pred_t)
print(f'PyTorch Linear — Test R²: {r2_pt_lin:.4f}')

## 🏗️ Model 2 — Deep Neural Network

In [ ]:
class DeepRegressor(nn.Module):
    """
    Multi-layer perceptron for regression.
    ReLU activations + BatchNorm + Dropout for regularisation.
    """
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)


deep_model = DeepRegressor(input_dim=X_tr_np.shape[1]).to(DEVICE)
params = sum(p.numel() for p in deep_model.parameters() if p.requires_grad)
print(f'Deep model parameters: {params:,}')

In [ ]:
deep_train_losses, deep_val_losses = train_model(
    deep_model, train_loader_v, X_val_t, y_val_t,
    epochs=300, lr=0.001, patience=20
)

deep_model.eval()
with torch.no_grad():
    y_pred_deep = deep_model(X_te_t).cpu().numpy().flatten()

r2_pt_deep = r2_score(y_te_np.flatten(), y_pred_deep)
print(f'PyTorch Deep — Test R²: {r2_pt_deep:.4f}')

## 📊 Summary: All Models Compared

In [ ]:
from sklearn.linear_model import LinearRegression as SkLR
sk = SkLR().fit(X_tr_np, y_tr_np.flatten())
r2_sk = r2_score(y_te_np.flatten(), sk.predict(X_te_np))

comparison = pd.DataFrame([
    {'Framework': 'sklearn OLS',      'R²': r2_sk},
    {'Framework': 'PyTorch Linear',   'R²': r2_pt_lin},
    {'Framework': 'PyTorch Deep MLP', 'R²': r2_pt_deep},
])
print(comparison.to_string(index=False))

# Loss comparison plot
plt.figure(figsize=(9, 4))
plt.plot(deep_train_losses, label='Deep Train MSE')
plt.plot(deep_val_losses,   label='Deep Val MSE')
plt.plot(train_losses,      label='Linear Train MSE', linestyle='--')
plt.plot(val_losses,        label='Linear Val MSE', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.yscale('log')
plt.title('Loss Curves: Linear vs Deep Model')
plt.legend()
plt.tight_layout()
plt.show()

## 💾 Save the Model

In [ ]:
import os
os.makedirs('models', exist_ok=True)

# Save only the weights (recommended for production)
torch.save(deep_model.state_dict(), 'models/pytorch_deep_regressor.pt')
print('✅ Model weights saved')

# Reload
loaded = DeepRegressor(input_dim=X_tr_np.shape[1]).to(DEVICE)
loaded.load_state_dict(torch.load('models/pytorch_deep_regressor.pt', map_location=DEVICE))
loaded.eval()
with torch.no_grad():
    y_check = loaded(X_te_t).cpu().numpy().flatten()
assert np.allclose(y_pred_deep, y_check, atol=1e-5), 'Mismatch!'
print('✅ Loaded model matches')

## 🎯 Summary

| Step | Code Pattern | Why |
|---|---|---|
| `optimizer.zero_grad()` | Before every batch | Prevents gradient accumulation |
| `loss.backward()` | After forward pass | Computes gradients via autograd |
| `optimizer.step()` | After backward | Updates parameters |
| `model.eval()` + `torch.no_grad()` | During validation | Disables dropout/BN training behaviour |
| `model.train()` | At start of training loop | Re-enables dropout/BN |

---

**Next:** [09_logistic_regression_sklearn.ipynb](09_logistic_regression_sklearn.ipynb)